# OpenAI weekly commentary

This notebook builds the latest fictional fact report, inspects the exact no-cost dry-run request, and keeps the single real API call behind an explicit flag. It never displays the API key.

In [ ]:
import json
from pathlib import Path
import sys
import tempfile

from IPython.display import JSON, Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from kcdk.commentary import (
    MemberCommentaryContext,
    generate_weekly_commentary,
    render_discord_markdown,
)
from kcdk.facts import build_weekly_fact_report
from kcdk.persistence import connect_database, import_week

In [ ]:
temporary_directory_context = tempfile.TemporaryDirectory(prefix='kcdk-commentary-')
database_path = Path(temporary_directory_context.name) / 'commentary.sqlite'
connection = connect_database(database_path)

for week in range(1, 5):
    import_week(
        connection,
        ROOT / 'data' / 'mock' / f'season_week_{week}.csv',
        ROOT / 'data' / 'mock' / 'members.csv',
        season_name='Mock 2026',
        season_identifier='mock-2026',
        season_year=2026,
        week_number=week,
        contest_name=f'Fictional Week {week}',
        contest_date=f'2026-09-{week:02d}',
    )

In [ ]:
report = build_weekly_fact_report(connection, 'mock-2026', max_facts=6)
display(JSON(report.to_dict()))

In [ ]:
safe_member_context = {
    'Casey North': MemberCommentaryContext(
        display_name='Casey North',
        nickname='North Star',
        commentary_notes='Space puns are welcome.',
    )
}
dry_run = generate_weekly_commentary(
    report, tone='normal', member_context=safe_member_context, dry_run=True
)
display(JSON(dry_run.to_dict()))

In [ ]:
print(
    f'Exact request size: {dry_run.request.prompt_characters:,} characters / '
    f'{dry_run.request.prompt_bytes:,} UTF-8 bytes; '
    f'{len(dry_run.request.fact_ids)} selected facts.'
)
print(json.dumps(dry_run.request.api_arguments(), indent=2, sort_keys=True))

## Optional one-request smoke test

Leave `RUN_LIVE_REQUEST` false during routine notebook execution. Set it to true only when you deliberately want one billable Responses API request and have configured the ignored `.env` file.

In [ ]:
RUN_LIVE_REQUEST = False
live_result = None
if RUN_LIVE_REQUEST:
    live_result = generate_weekly_commentary(
        report, tone='normal', member_context=safe_member_context
    )
    display(JSON(live_result.to_dict()))
else:
    print('Live request skipped. Set RUN_LIVE_REQUEST = True to make exactly one request.')

In [ ]:
if live_result is not None and live_result.commentary is not None:
    display(JSON(live_result.commentary.to_dict()))
    display(Markdown(render_discord_markdown(live_result.commentary)))
else:
    print('Structured commentary and Discord Markdown will appear here after the optional live request.')

In [ ]:
connection.close()
temporary_directory_context.cleanup()